# PRO-cap Atlas BPNet locus viewer

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kundajelab/procap-atlas/blob/main/notebooks/procap_atlas_bpnet_locus_viewer.ipynb)

This lightweight notebook visualizes one locus with fold-averaged BPNet predictions and DeepLIFT/SHAP logos. It uses the production-style genomic nucleotide-frequency soft reference from `src/bpnet/attribute/attribute_bpnet.py` and leaves shuffled-reference instability diagnostics to `procap_atlas_bpnet_locus_diagnostics.py`.


In [ ]:
from pathlib import Path as _Path
import importlib.util as _importlib_util
import os as _os
import sys as _sys

_numpy_spec = _importlib_util.find_spec("numpy")
_numpy_origin = _Path(_numpy_spec.origin).resolve() if _numpy_spec else None
_environment_root = _Path(_sys.prefix).resolve()
print(f"Python: {_sys.executable}")
print(f"Environment: {_environment_root}")
print(f"NumPy candidate: {_numpy_origin}")
print(f"PYTHONPATH: {_os.environ.get('PYTHONPATH')!r}")
if _os.environ.get("PYTHONPATH"):
    raise RuntimeError(
        "The PRO-cap Atlas kernel did not remove the Open OnDemand PYTHONPATH. "
        "Reinstall it with notebooks/install_uv_kernel.py and restart JupyterLab."
    )
_ondemand_paths = [
    path for path in _sys.path if path.startswith("/share/software/user/open/py-jupyterlab/")
]
if _ondemand_paths:
    raise RuntimeError(
        f"Open OnDemand PYTHONPATH entries are affecting imports: {_ondemand_paths}. "
        "Reinstall the kernel with notebooks/install_uv_kernel.py."
    )
if _numpy_origin is None or not _numpy_origin.is_relative_to(_environment_root):
    raise RuntimeError(
        "NumPy is not resolving from the selected uv environment. Reinstall the "
        "PRO-cap Atlas kernel with notebooks/install_uv_kernel.py."
    )


In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import seaborn as sns
import torch

for _parent in [Path.cwd(), *Path.cwd().parents]:
    if (_parent / "src").is_dir() and str(_parent) not in sys.path:
        sys.path.insert(0, str(_parent))
        break

from notebooks.locus_viewer import (
    deeplift_attributions,
    ensemble_prediction,
    logo_offsets_for_locus,
    locus_input,
    plot_locus_summary,
    save_locus_viewer_outputs,
    setup_experiment,
)

sns.set_style("whitegrid")


## Configuration

Choose the experiment, point locus, displayed prediction interval, and logo interval. Coordinates are 1-based in the configuration strings.


In [ ]:
EXP_ID = "ENCSR342WAR"
POINT_REGION = "chr2:181680717"
VIEW_REGION = "chr2:181680467-181681166"
LOGO_REGION = "chr2:181680467-181681167"
REVERSE_COMPLEMENT = False
N_FOLDS = 7
BATCH_SIZE = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK_DIR = Path(os.environ.get("SCRATCH", ".cache")) / "procap_atlas_locus_viewer"
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE}")
print(f"Work directory: {WORK_DIR}")


In [ ]:
resources = setup_experiment(EXP_ID, WORK_DIR, N_FOLDS)
chrom, center, X = locus_input(resources, POINT_REGION)
logo_chrom, logo_start, logo_end, logo_offsets = logo_offsets_for_locus(
    POINT_REGION, LOGO_REGION
)
print(resources["config"].get("biosample", EXP_ID), X.shape, logo_offsets)


## Predictions

Run each fold locally and average the count-scaled strand profiles. The observed and predicted tracks are plotted separately in the combined summary figure below so each can use its own y scale.


In [ ]:
prediction = ensemble_prediction(resources, X, N_FOLDS, DEVICE)


## Frequency-reference DeepLIFT and summary figure

Compute profile-head and count-head DeepLIFT/SHAP logos using one soft reference whose A/C/G/T probabilities are the observed input-wide nucleotide frequencies. The observed track, predicted track, and logos are stacked in one figure to make the panels easier to compare.


In [ ]:
attributions = deeplift_attributions(
    resources, X, logo_offsets, N_FOLDS, BATCH_SIZE, DEVICE
)
plot_locus_summary(
    prediction,
    attributions,
    resources,
    EXP_ID,
    POINT_REGION,
    VIEW_REGION,
    LOGO_REGION,
    logo_start,
    logo_end,
    REVERSE_COMPLEMENT,
)


## Optional save

Save the current figures and arrays for later inspection.


In [ ]:
OUTPUT_DIR = (
    Path("plots/bpnet/locus_viewer")
    / EXP_ID
    / POINT_REGION.replace(":", "_").replace(",", "")
)
save_locus_viewer_outputs(
    OUTPUT_DIR,
    prediction,
    attributions,
    resources,
    EXP_ID,
    POINT_REGION,
    VIEW_REGION,
    LOGO_REGION,
    logo_start,
    logo_end,
    REVERSE_COMPLEMENT,
)
print(f"Saved {OUTPUT_DIR}")
